In [52]:
# EVALUATION CELL 1: Network Skeletons and Graph Utilities
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gym
import cityflow
import gym_cityflow
from torch_geometric.nn import GATConv
from pathlib import Path
import os
import random

# Hardware config
eval_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Basic Brain (For DQN and Double DQN) ---
class TrafficIntersectionBrain(nn.Module):
    def __init__(self, features_in=72, num_actions=9, layer_widths=[128, 256, 128]):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(features_in, layer_widths[0]),
            nn.LayerNorm(layer_widths[0]),
            nn.ReLU(),
            nn.Linear(layer_widths[0], layer_widths[1]),
            nn.LayerNorm(layer_widths[1]),
            nn.ReLU(),
            nn.Linear(layer_widths[1], layer_widths[2]),
            nn.LayerNorm(layer_widths[2]),
            nn.ReLU(),
            nn.Linear(layer_widths[2], num_actions)
        )
    def forward(self, x, edge_idx=None): # edge_idx ignored here for compatibility
        return self.network(x)

# --- 2. Dueling Brain (For D3QN) ---
class BifurcatedTrafficBrain(nn.Module):
    def __init__(self, features_in=72, num_actions=9, shared_dims=[128, 256], split_dim=128):
        super().__init__()
        self.core_processor = nn.Sequential(
            nn.Linear(features_in, shared_dims[0]),
            nn.LayerNorm(shared_dims[0]),
            nn.ReLU(),
            nn.Linear(shared_dims[0], shared_dims[1]),
            nn.LayerNorm(shared_dims[1]),
            nn.ReLU()
        )
        self.value_head = nn.Sequential(
            nn.Linear(shared_dims[1], split_dim), nn.LayerNorm(split_dim), nn.ReLU(), nn.Linear(split_dim, 1)
        )
        self.advantage_head = nn.Sequential(
            nn.Linear(shared_dims[1], split_dim), nn.LayerNorm(split_dim), nn.ReLU(), nn.Linear(split_dim, num_actions)
        )
    def forward(self, x, edge_idx=None):
        core = self.core_processor(x)
        val = self.value_head(core)
        adv = self.advantage_head(core)
        return val + (adv - adv.mean(dim=1, keepdim=True))

# --- 3. Graph Attention Brain (For GAT and PER-GAT) ---
class AttentionGraphTrafficBrain(nn.Module):
    def __init__(self, features_in=72, num_actions=9, shared_dims=[64, 64], split_dim=32, heads=4):
        super().__init__()
        self.gat_layer_1 = GATConv(features_in, shared_dims[0], heads=heads, concat=True)
        self.gat_layer_2 = GATConv(shared_dims[0] * heads, shared_dims[1], heads=1, concat=False)
        self.value_head = nn.Sequential(
            nn.Linear(shared_dims[1], split_dim), nn.LayerNorm(split_dim), nn.ReLU(inplace=True), nn.Linear(split_dim, 1)
        )
        self.advantage_head = nn.Sequential(
            nn.Linear(shared_dims[1], split_dim), nn.LayerNorm(split_dim), nn.ReLU(inplace=True), nn.Linear(split_dim, num_actions)
        )
    def forward(self, x, edge_idx):
        h = F.elu(self.gat_layer_1(x, edge_idx))
        h = F.elu(self.gat_layer_2(h, edge_idx))
        v = self.value_head(h)
        a = self.advantage_head(h)
        return v + (a - a.mean(dim=1, keepdim=True))

# --- 4. Strict Edge Builder ---
def extract_strict_edges(config_loc, active_nodes):
    config = json.load(open(config_loc, "r"))
    net_path = config["roadnetFile"] if "dir" not in config else os.path.join(config["dir"], config["roadnetFile"])
    road_data = json.load(open(net_path, "r"))
    
    mapping = {n: i for i, n in enumerate(active_nodes)}
    connections = []
    for r in road_data["roads"]:
        s, e = r.get("startIntersection"), r.get("endIntersection")
        if s in mapping and e in mapping:
            connections.extend([[mapping[s], mapping[e]], [mapping[e], mapping[s]]])
    
    return torch.tensor(connections, dtype=torch.long).t().contiguous().to(eval_device)

# Add this to the bottom of EVALUATION CELL 1
def build_max_pressure_map(config_loc, active_nodes):
    """Parses the roadnet JSON to map exact incoming/outgoing lanes for every phase."""
    config = json.load(open(config_loc, "r"))
    net_path = config["roadnetFile"] if "dir" not in config else os.path.join(config["dir"], config["roadnetFile"])
    roadnet = json.load(open(net_path, "r"))
    
    mp_map = {}
    for inter in roadnet['intersections']:
        node_id = inter['id']
        if node_id not in active_nodes:
            continue
            
        road_links = inter['roadLinks']
        phases = inter['trafficLight']['lightphases']
        
        phase_dict = {}
        for p_idx, phase in enumerate(phases):
            in_lanes = []
            out_lanes = []
            for link_idx in phase['availableRoadLinks']:
                link = road_links[link_idx]
                start_road = link['startRoad']
                end_road = link['endRoad']
                for lane_link in link['laneLinks']:
                    in_lanes.append(f"{start_road}_{lane_link['startLaneIndex']}")
                    out_lanes.append(f"{end_road}_{lane_link['endLaneIndex']}")
            phase_dict[p_idx] = {'in': in_lanes, 'out': out_lanes}
        mp_map[node_id] = phase_dict
        
    return mp_map

In [51]:
# EVALUATION CELL 2: The Exact Reference Engine (With Max Pressure)
import random

def run_scientific_evaluation(config_file, weight_file, strategy="gat_d3qn", steps=1000):
    print(f"\n--- Launching Evaluation: {strategy.upper()} ---")
    
    out_dir = Path("evaluation_results")
    out_dir.mkdir(exist_ok=True)
    
    sim_env = gym.make(id='cityflow-v0', configPath=config_file, episodeSteps=steps)
    initial_snapshot = sim_env.reset()
    controlled_intersections = sorted(initial_snapshot.keys())
    agent_count = len(controlled_intersections)
    action_dim = 9 
    
    replay_file_name = str(out_dir / f"replay_{strategy}.txt")
    sim_env.eng.set_save_replay(True)
    sim_env.eng.set_replay_file(replay_file_name)
    print(f"-> Recording web frontend video to: {replay_file_name}")
    
    city_graph_edges = None
    policy = None
    mp_map = None

    if strategy in ["dqn", "ddqn"]:
        policy = TrafficIntersectionBrain().to(eval_device)
    elif strategy == "d3qn":
        policy = BifurcatedTrafficBrain().to(eval_device)
    elif strategy in ["gat_d3qn", "per_gat_d3qn"]:
        policy = AttentionGraphTrafficBrain().to(eval_device)
        city_graph_edges = extract_strict_edges(config_file, controlled_intersections)
    elif strategy == "max_pressure":
        mp_map = build_max_pressure_map(config_file, controlled_intersections)
    
    if strategy not in ["fixed_time", "random", "max_pressure"]:
        weights = torch.load(weight_file, map_location=eval_device, weights_only=True)
        policy.load_state_dict(weights)
        policy.eval() 
        print(f"-> Weights successfully loaded from {weight_file}")
    else:
        print(f"-> Using Algorithmic Baseline: {strategy.upper()}")

    episode_reward = 0
    accumulated_waiting_time = 0
    total_accumulated_reward = 0.0
    arrived_vehicles = set()
    prev_vehicles = set(sim_env.eng.get_vehicles(include_waiting=True))
    step_stats = []

    current_obs = [np.array(initial_snapshot[node], dtype=np.float32).flatten() for node in controlled_intersections]
    mp_current_actions = [1] * agent_count # Default starting phase for MP
    
    for t in range(steps):
        actions = []
        
        if strategy == "fixed_time":
            for i in range(agent_count):
                row = i // 4  
                col = i % 4   
                offset = (row + col) * 10  
                phase = ((t + offset) // 30) % action_dim
                actions.append(phase)
                
        elif strategy == "random":
            actions = [random.randint(0, action_dim - 1) for _ in range(agent_count)]
            
        elif strategy == "max_pressure":
            # Re-evaluate pressure every 10 seconds to prevent phase-flickering
            if t % 10 == 0 or t == 0:
                actions = []
                # Use absolute vehicle counts for pressure
                lane_veh = sim_env.eng.get_lane_vehicle_count() 
                for node in controlled_intersections:
                    best_phase = 1
                    max_p = -99999
                    # Only evaluate the 4 main green phases
                    for p in [1, 2, 3, 4]:
                        in_count = sum(lane_veh.get(l, 0) for l in mp_map[node][p]['in'])
                        out_count = sum(lane_veh.get(l, 0) for l in mp_map[node][p]['out'])
                        pressure = in_count - out_count
                        if pressure > max_p:
                            max_p = pressure
                            best_phase = p
                    actions.append(best_phase)
                mp_current_actions = actions
            else:
                actions = mp_current_actions
                
        else: # Neural Network RL Agents
            state_tensor = torch.tensor(np.array(current_obs), dtype=torch.float32, device=eval_device)
            with torch.no_grad():
                q_forecasts = policy(state_tensor, city_graph_edges)
            actions = q_forecasts.argmax(dim=1).cpu().numpy().tolist()

        # Step the physical engine
        next_raw_obs, rewards, done, _ = sim_env.step(actions)
        
        reward_dict  = {k:v for k,v in rewards}
        rewards_list = [reward_dict[k] for k in controlled_intersections]
        step_r = sum(rewards_list)
        episode_reward += step_r
        total_accumulated_reward += step_r
        
        current_obs = [np.array(next_raw_obs[k], dtype=np.float32).flatten() for k in controlled_intersections]

        cur_vehicles = set(sim_env.eng.get_vehicles(include_waiting=True))
        new_arrivals = prev_vehicles - cur_vehicles
        arrived_vehicles |= new_arrivals
        prev_vehicles = cur_vehicles
        
        num_arrived = len(arrived_vehicles)
        active_vehicles = sim_env.eng.get_vehicle_count()
        avg_travel_time = sim_env.eng.get_average_travel_time()
        
        lane_wait = sim_env.eng.get_lane_waiting_vehicle_count()
        current_total_wait = sum(lane_wait.values())
        accumulated_waiting_time += current_total_wait
        
        avg_wait_per_veh = 0.0
        if num_arrived > 0:
             avg_wait_per_veh = accumulated_waiting_time / num_arrived
        
        step_stats.append({
            "step": t + 1,
            "total_reward": total_accumulated_reward,
            "num_arrived": num_arrived,
            "active_vehicles": active_vehicles,
            "avg_wait_time": float(f"{avg_wait_per_veh:.2f}"),
            "avg_travel_time": float(f"{avg_travel_time:.2f}")
        })
        
        if done:
            break

    sim_env.close() 
    
    out_file = out_dir / f"metrics_{strategy}.json"
    with open(out_file, 'w') as f:
        json.dump(step_stats, f, indent=4)
        
    print(f"-> Evaluation Complete! Final Score: {episode_reward}")
    print(f"-> Cars Processed: {num_arrived}")
    
    return step_stats

In [53]:
stats_base = run_scientific_evaluation(config_target, None, strategy="random")
stats_mp = run_scientific_evaluation("../configs/Intersections_4/sample_config.json", None, strategy="max_pressure", steps=1000)


--- Launching Evaluation: RANDOM ---
-> Recording web frontend video to: evaluation_results/replay_random.txt
-> Using Algorithmic Baseline: RANDOM
-> Evaluation Complete! Final Score: -954027.5999999995
-> Cars Processed: 2346

--- Launching Evaluation: MAX_PRESSURE ---
-> Recording web frontend video to: evaluation_results/replay_max_pressure.txt
-> Using Algorithmic Baseline: MAX_PRESSURE
-> Evaluation Complete! Final Score: -1465105.8000000005
-> Cars Processed: 2387


In [36]:
# EVALUATION CELL 3: Run the Tests!
config_target = "../configs/Intersections_4/sample_config.json"

# 1. The Algorithmic Baseline (No weights needed)
stats_base = run_scientific_evaluation(config_target, None, strategy="fixed_time")

# 2. Standard DQN
stats_dqn = run_scientific_evaluation(config_target, "dqn/intersection_brain_final.pth", strategy="dqn")

# 3. Double DQN
stats_ddqn = run_scientific_evaluation(config_target, "ddqn/intersection_brain_final.pth", strategy="ddqn")

# 4. Double Dueling DQN
stats_d3qn = run_scientific_evaluation(config_target, "duel_ddqn/intersection_brain_final.pth", strategy="d3qn")

# 5. Graph Attention D3QN
stats_gat = run_scientific_evaluation(config_target, "duel_ddqn_gat/d3qn_gat.pth", strategy="gat_d3qn")

# 6. Prioritized Graph Attention D3QN 
# (Note: Inference for PER is mathematically identical to GAT, we just load the PER-trained weights!)
stats_per = run_scientific_evaluation(config_target, "d3qn_per/per_no_loop_d3qn_gat.pth", strategy="per_gat_d3qn")


--- Launching Evaluation: FIXED_TIME ---
-> Recording web frontend video to: evaluation_results/replay_fixed_time.txt
-> Using Algorithmic Green-Wave Baseline.
-> Evaluation Complete! Final Score: -4269235.2
-> Cars Processed: 1606
-> Metrics saved to: evaluation_results/metrics_fixed_time.json


--- Launching Evaluation: DQN ---
-> Recording web frontend video to: evaluation_results/replay_dqn.txt
-> Weights successfully loaded from dqn/intersection_brain_final.pth
-> Evaluation Complete! Final Score: -249989.39999999985
-> Cars Processed: 2541
-> Metrics saved to: evaluation_results/metrics_dqn.json


--- Launching Evaluation: DDQN ---
-> Recording web frontend video to: evaluation_results/replay_ddqn.txt
-> Weights successfully loaded from ddqn/intersection_brain_final.pth
-> Evaluation Complete! Final Score: -275215.1999999999
-> Cars Processed: 2545
-> Metrics saved to: evaluation_results/metrics_ddqn.json


--- Launching Evaluation: D3QN ---
-> Recording web frontend video to: e

In [57]:
import json
import os
from pathlib import Path

# Define the models we want to evaluate
models = ["max_pressure","random", "dqn", "ddqn", "d3qn", "gat_d3qn", "per_gat_d3qn"]
labels = ["Max Pressure", "Fixed Time (Baseline)", "Standard DQN", "Double DQN", "D3QN", "GAT-D3QN", "PER-GAT-D3QN"]
results_dir = Path("evaluation_results")

# Dictionary to hold the final step data for each model
final_stats = {}

# 1. Extract the last step from every JSON file
for mod, label in zip(models, labels):
    file_path = results_dir / f"metrics_{mod}.json"
    if not file_path.exists():
        continue
        
    with open(file_path, "r") as f:
        data = json.load(f)
        
    # Grab the very last tick of the simulation
    final_stats[mod] = {
        "label": label,
        "arrived": data[-1]["num_arrived"],
        "active": data[-1]["active_vehicles"],
        "wait": data[-1]["avg_wait_time"],
        "travel": data[-1]["avg_travel_time"]
    }

# 2. Extract Baseline values for percentage calculation
if "random" not in final_stats:
    print("Error: Run the random evaluation first to generate baseline comparisons.")
else:
    base_wait = final_stats["random"]["wait"]
    base_travel = final_stats["random"]["travel"]
    base_active = final_stats["random"]["active"]

    # 3. Print the Markdown Table
    # Header
    print("\n" + "="*90)
    print(f"{'Model':<25} {'Arrived':<10} {'Active':<20} {'Avg Wait (s)':<20} {'Avg Travel (s)':<20}")
    print("="*90)

    for mod in models:
        if mod not in final_stats:
            continue
            
        stats = final_stats[mod]

        if mod == "fixed_time":
            print(f"{stats['label']:<25} "
                f"{stats['arrived']:<10} "
                f"{stats['active']:<20} "
                f"{stats['wait']:<20.2f} "
                f"{stats['travel']:<20.2f}")
        else:
            # Improvements
            wait_imp = ((base_wait - stats['wait']) / base_wait) * 100 if base_wait > 0 else 0
            travel_imp = ((base_travel - stats['travel']) / base_travel) * 100 if base_travel > 0 else 0
            active_imp = ((base_active - stats['active']) / base_active) * 100 if base_active > 0 else 0

            # Format strings
            wait_str = f"{stats['wait']:.2f} ({-wait_imp:+.1f}%)"
            travel_str = f"{stats['travel']:.2f} ({-travel_imp:+.1f}%)"
            active_str = f"{stats['active']} ({-active_imp:+.1f}%)"

            print(f"{stats['label']:<25} "
                f"{stats['arrived']:<10} "
                f"{active_str:<20} "
                f"{wait_str:<20} "
                f"{travel_str:<20}")

    print("="*90)


Model                     Arrived    Active               Avg Wait (s)         Avg Travel (s)      
Max Pressure              2387       728 (-2.4%)          73.65 (+42.6%)       186.19 (+0.8%)      
Fixed Time (Baseline)     2346       746 (-0.0%)          51.64 (-0.0%)        184.78 (-0.0%)      
Standard DQN              2541       574 (-23.1%)         12.06 (-76.6%)       153.87 (-16.7%)     
Double DQN                2545       570 (-23.6%)         13.75 (-73.4%)       155.78 (-15.7%)     
D3QN                      2561       554 (-25.7%)         10.07 (-80.5%)       151.75 (-17.9%)     
GAT-D3QN                  2540       575 (-22.9%)         12.88 (-75.1%)       154.76 (-16.2%)     
PER-GAT-D3QN              2543       572 (-23.3%)         13.35 (-74.1%)       154.71 (-16.3%)     
